In [ ]:
# Parameters
EPOCHS = 25
BATCH_SIZE = 32
SKIP_YOLO = False
FRAMES_PER_VIDEO = 10


# Celeb-DF v2 Training Pipeline (Kaggle GPU)

Trains **EfficientNet-B3**, **ConvNeXt-Tiny**, and **YOLO26n-cls** on the official Celeb-DF v2 benchmark.

**Inputs**:
- Dataset: `reubensuju/celeb-df-v2` (mounted at `/kaggle/input/celeb-df-v2`)
- Code: `mradulmanimishra/deepfake-code` (mounted at `/kaggle/input/deepfake-code`)


In [ ]:
# ── Install dependencies ──────────────────────────────────────
# Kaggle images already ship torch/torchvision/timm/ultralytics.
# Only attempt a network install for what is genuinely missing, and
# never fail the kernel when the sandbox has no internet access.
import importlib, subprocess, sys

REQUIRED = ['timm', 'ultralytics', 'facenet_pytorch', 'sklearn', 'cv2']
missing = []
for mod in REQUIRED:
    try:
        importlib.import_module(mod)
    except Exception:
        missing.append(mod)

if missing:
    print(f'Missing packages: {missing} - attempting pip install')
    try:
        subprocess.run(
            [sys.executable, '-m', 'pip', 'install', '-q',
             'timm>=1.0.0', 'ultralytics>=8.2.0',
             'facenet-pytorch>=2.6.0', 'scikit-learn>=1.3.0'],
            check=False, timeout=600)
    except Exception as e:
        print(f'[warn] pip install failed (no internet?): {e}')
else:
    print('All required packages already present.')

for mod in REQUIRED:
    try:
        importlib.import_module(mod)
        print(f'  [ok] {mod}')
    except Exception as e:
        print(f'  [MISSING] {mod}: {e}')


In [ ]:
# ── Mount & Extract Codebase ──────────────────────────────────
import os, sys, zipfile, shutil
from pathlib import Path

REPO_DIR = Path('/kaggle/working')
CODE_INPUT = Path('/kaggle/input/deepfake-code')

# The dataset arrives either as a zip (older upload) or as an
# already-extracted directory tree (current upload). Support both.
code_zip = CODE_INPUT / 'deepfake_code.zip'

if code_zip.exists():
    with zipfile.ZipFile(code_zip, 'r') as zf:
        zf.extractall(str(REPO_DIR))
    print('Code unpacked from deepfake-code dataset (zip).')
else:
    # Already extracted: find the directory that contains config.py
    candidates = [CODE_INPUT] + sorted(p for p in CODE_INPUT.glob('*') if p.is_dir())
    extracted_root = None
    for cand in candidates:
        if (cand / 'config.py').exists() and (cand / 'main.py').exists():
            extracted_root = cand
            break
    if extracted_root is None:
        raise FileNotFoundError(
            f'Could not locate the codebase under {CODE_INPUT}. '
            f'Contents: {sorted(p.name for p in CODE_INPUT.iterdir())[:20]}')
    shutil.copytree(extracted_root, REPO_DIR, dirs_exist_ok=True)
    print(f'Code copied from {extracted_root} (extracted dataset layout).')

sys.path.insert(0, str(REPO_DIR))
os.chdir(str(REPO_DIR))
print(f'Working directory: {Path.cwd()}')
assert (REPO_DIR / 'config.py').exists(), 'config.py missing after mount'
print('Codebase mount OK.')


In [ ]:
# ── Verify GPU ────────────────────────────────────────────────
import torch
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU Device: {torch.cuda.get_device_name(0)}')
else:
    print('⚠️ Running on CPU')


In [ ]:
# ── Extract Celeb-DF Faces ─────────────────────────────────────
import config
from preprocessing.extract_faces_from_videos import VideoFaceExtractor
from utils.helpers import setup_logger, ensure_dirs

celeb_input = Path('/kaggle/input/celeb-df-v2')
if not celeb_input.exists():
    # Fallback to subdirectory if nested
    candidates = list(Path('/kaggle/input').glob('*celeb*'))
    if candidates:
        celeb_input = candidates[0]

print(f'Celeb-DF source: {celeb_input}')
out_dir = REPO_DIR / 'data' / 'processed' / 'celebdf'
ensure_dirs(out_dir)

logger = setup_logger('CelebDF_Extract')
extractor = VideoFaceExtractor(
    frames_per_video     = int(FRAMES_PER_VIDEO),
    image_size           = 224,
    margin_ratio         = 0.20,
    min_face_prob        = 0.90,
    fallback_center_crop = True,
    device               = config.DEVICE,
    logger               = logger
)

extractor.process_and_split(
    video_dir          = celeb_input,
    output_dir         = out_dir,
    split_mode         = 'celeb_df',
    clean_output       = False,
    seed               = 42
)
print('✅ Face extraction and official benchmark split complete.')


In [ ]:
# ── Train All 3 Models on Celeb-DF ─────────────────────────────
import types
from scripts.train_all_datasets import run_all

train_args = types.SimpleNamespace(
    datasets   = ['celebdf'],
    epochs     = int(EPOCHS),
    batch_size = int(BATCH_SIZE),
    patience   = 7,
    pilot      = False,
    skip_yolo  = bool(SKIP_YOLO),
    force      = True,
)

print(f'Starting Celeb-DF training grid for {EPOCHS} epochs...')
run_all(train_args)
print('✅ Celeb-DF models trained successfully.')


In [ ]:
# ── Evaluate Ensemble on Celeb-DF Test Split ───────────────────
import subprocess
subprocess.check_call(['python', 'main.py', 'evaluate-ensemble', '--dataset', 'celebdf'])
subprocess.check_call(['python', 'main.py', 'generate-reports', '--allow-toy'])
print('✅ Evaluation complete.')


In [ ]:
# ── Package Checkpoints & Reports ──────────────────────────────
import shutil

ckpt_archive = REPO_DIR / 'celebdf_checkpoints'
shutil.make_archive(str(ckpt_archive), 'zip', root_dir=str(REPO_DIR / 'models'), base_dir='checkpoints')

reports_archive = REPO_DIR / 'celebdf_reports'
shutil.make_archive(str(reports_archive), 'zip', root_dir=str(REPO_DIR), base_dir='reports')

print(f'✅ Checkpoints archived → {ckpt_archive}.zip')
print(f'✅ Reports archived     → {reports_archive}.zip')
